<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 1: Building Reliable AI Agents

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Get structured outputs from LLMs** — force the LLM to return data in a predictable format
2. **Build an LLM-as-Judge evaluator** — use one LLM to check another's work

---

## 1. Environment Setup

In [ ]:
!pip install -q openai pydantic

In [ ]:
import os
import json
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel

In [ ]:
api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

---

## 2. The Problem — LLM Outputs Are Unpredictable

LLMs are powerful, but their outputs are **inconsistent**. Ask the same extraction question three times and you might get three different formats:

| Attempt | LLM Response |
|---|---|
| 1 | "Name: Priya Sharma, City: Bhubaneswar, Email: priya@techsolutions.in" |
| 2 | "• **Name:** Priya Sharma\n• **City:** Bhubaneswar\n• **Email:** priya@techsolutions.in" |
| 3 | "The person mentioned is Priya Sharma from Bhubaneswar. Her email is priya@techsolutions.in." |

All three contain the same information — but if your code needs to extract the name, city, or email programmatically, **none of these formats are reliable**. This is a real problem when building production systems.

Let's see this inconsistency in action:

In [ ]:
# Ask the LLM to extract structured info — watch the format change each time
text = "Priya Sharma from Bhubaneswar recently joined as CTO. Reach her at priya@techsolutions.in."

for i in range(3):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"Extract the person's name, city, and email from this text:\n\n{text}"}],
        temperature=1.0
    )
    answer = response.choices[0].message.content
    print(f"Attempt {i+1}:\n{answer}\n{'-'*40}")

    # Try to use it programmatically — parsing free text is fragile
    try:
        data = json.loads(answer)
        print(f"  ✓ Parsed as JSON: {data}\n")
    except json.JSONDecodeError:
        print(f"  ✗ Not valid JSON — can't use this programmatically\n")

Each attempt returns the **same information** but in a **different format** — bullet points, prose, key-value pairs, etc. Since the output isn't valid JSON, we can't reliably extract `name`, `city`, or `email` in code. We need a way to **guarantee** the output format. That's where structured outputs come in.

---

## 3. Structured Outputs with Pydantic

### The "Form vs. Letter" Analogy

Imagine asking someone for their contact info:

- **Letter** (regular LLM): "Hi, I'm Priya from Bhubaneswar. You can reach me at priya@email.com or call 98765..." — free text, hard to extract fields
- **Form** (structured output): Name: `Priya` | City: `Bhubaneswar` | Email: `priya@email.com` — every field in its place

**Pydantic** lets us define a "form" that the LLM must fill out. The API **guarantees** the response matches our structure.

### How It Works

```
┌─────────────┐     ┌──────────────┐     ┌─────────────────┐
│  Your Code  │────>│  OpenAI API  │────>│  Pydantic Model │
│             │     │              │     │                 │
│ "Extract    │     │ Forces LLM   │     │ name: "Priya"   │
│  contact    │     │ to fill the  │     │ city: "Bhub..."  │
│  info"      │     │ form exactly │     │ email: "p@..."   │
└─────────────┘     └──────────────┘     └─────────────────┘
```

Let's solve our extraction problem from Section 2:

In [ ]:
# Define our "form" — the structure we want the LLM to return
class ContactInfo(BaseModel):
    """Structured contact information."""
    name: str
    city: str
    email: str

# Same text from Section 2 — but now with guaranteed structure
text = "Priya Sharma from Bhubaneswar recently joined as CTO. Reach her at priya@techsolutions.in."

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the person's name, city, and email from this text:\n\n{text}"}],
    response_format=ContactInfo
)

contact = response.choices[0].message.parsed
print(f"Name:  {contact.name}")
print(f"City:  {contact.city}")
print(f"Email: {contact.email}")
print(f"\nAs dict: {contact.model_dump()}")

### Regular vs. Structured API Call

| | Regular Call | Structured Call |
|---|---|---|
| **Method** | `client.chat.completions.create()` | `client.beta.chat.completions.parse()` |
| **Returns** | Free text string | Pydantic object with typed fields |
| **Format** | Unpredictable | Guaranteed to match your model |
| **Access** | `response.choices[0].message.content` | `response.choices[0].message.parsed` |

### Pydantic Essentials for Structured Outputs

`ContactInfo` used only `str` fields — but Pydantic gives you much more. You can constrain types, set numeric ranges, limit choices, and add descriptions that help the LLM understand what you want. Let's explore these features.

In [ ]:
from pydantic import Field
from typing import Literal

class StudentProfile(BaseModel):
    name: str
    age: int = Field(gt=0, lt=120, description="Student's age in years")
    gpa: float = Field(ge=0.0, le=10.0, description="GPA on a 10-point scale")
    department: Literal["CSE", "ECE", "ME", "CE"]
    skills: list[str]

# Valid
student = StudentProfile(name="Rahul", age=21, gpa=8.5, department="CSE", skills=["Python", "ML"])
print(student.model_dump())

# Invalid — see what happens
try:
    bad = StudentProfile(name="Test", age=-5, gpa=12.0, department="Art", skills=[])
except Exception as e:
    print(f"\nValidation error:\n{e}")

In [ ]:
response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": """Extract student info from this text:

Ananya Patel is a 20-year-old Computer Science student at CV Raman University.
She maintains a 9.2 GPA and is skilled in Java, cloud computing, and data analysis."""}],
    response_format=StudentProfile
)

student = response.choices[0].message.parsed
print(f"Name: {student.name}")
print(f"Age: {student.age} (type: {type(student.age).__name__})")
print(f"GPA: {student.gpa}")
print(f"Department: {student.department}")
print(f"Skills: {student.skills}")

### Quick Reference — Pydantic Field Features

| Feature | Example | Why It Matters |
|---|---|---|
| `int`, `float`, `bool` | `age: int` | Right type, not a string |
| `Field(gt=0, lt=120)` | Numeric bounds (exclusive) | Prevents nonsense values |
| `Field(ge=, le=)` | `gpa: float = Field(ge=0, le=10)` | Inclusive bounds |
| `Literal[...]` | `department: Literal["CSE", "ECE"]` | LLM must pick from your list |
| `list[str]` | `skills: list[str]` | Extracts multiple items |
| `Field(description=...)` | Helps LLM understand the field | Better extraction accuracy |

### Beyond Data Extraction

Structured outputs aren't just for pulling names and emails out of text. **Any time you need a programmatic decision from an LLM** — a yes/no, a score, a classification — you need guaranteed structure.

In the next section, we'll use this same technique to build an automated quality checker: an LLM that evaluates another LLM's responses and returns a structured verdict.

---

## 4. LLM-as-Judge — Automated Quality Control

Now that we can get structured outputs, let's use this for something powerful: **having one LLM check another's work**.

The idea is simple — given a question and a response, ask the LLM: "Is this response acceptable?" and get back a structured yes/no with feedback.

In [ ]:
# Define the evaluation "form" — same pattern as ContactInfo
class Evaluation(BaseModel):
    """Structured evaluation result from the LLM judge."""
    is_acceptable: bool    # True = response passes quality check
    feedback: str          # Explanation of the evaluation

def evaluate_response(question, response) -> Evaluation:
    """Use the LLM to judge whether a response is acceptable."""
    result = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Evaluate if the response accurately and helpfully answers the question."},
            {"role": "user", "content": f"Question: {question}\n\nResponse: {response}\n\nIs this response acceptable?"}
        ],
        response_format=Evaluation
    )
    return result.choices[0].message.parsed

In [ ]:
# Test with a GOOD response
evaluation = evaluate_response(
    "What is the capital of India?",
    "The capital of India is New Delhi."
)
print("Testing a GOOD response:")
print(f"  is_acceptable: {evaluation.is_acceptable}")
print(f"  feedback: {evaluation.feedback}")

In [ ]:
# Test with a BAD response
evaluation = evaluate_response(
    "What is the capital of India?",
    "I think it might be Mumbai or something."
)
print("Testing a BAD response:")
print(f"  is_acceptable: {evaluation.is_acceptable}")
print(f"  feedback: {evaluation.feedback}")

---

## 5. Key Takeaways

1. **LLM outputs are unpredictable** — you can't trust free-text responses for structured data

2. **Pydantic + `.parse()` guarantees structure** — the API forces the LLM to fill out your exact form

3. **LLM-as-Judge automates quality control** — use one LLM to evaluate another's responses with structured verdicts

### Concept-to-Code Mapping

| Concept | Our Code | What It Does |
|---|---|---|
| **Structured Output** | `ContactInfo(BaseModel)` + `.parse()` | Forces LLM to return typed fields |
| **Evaluation Form** | `Evaluation(BaseModel)` + `.parse()` | Gets structured yes/no + feedback from judge |
| **Evaluator** | `evaluate_response()` | Judges response quality |

---

## 6. Exercises

### Exercise 1: Richer Evaluation Model (Beginner)
Add more fields to the `Evaluation` model (e.g., `accuracy_score: int`, `tone_score: int`). Test the updated evaluator with different question-response pairs.

### Exercise 2: Domain-Specific Judge (Intermediate)
Modify `evaluate_response()` to accept a custom system prompt with domain-specific criteria (e.g., "must cite sources", "must be under 50 words"). Test with responses that pass and fail your criteria.

### Exercise 3: Build a Pydantic Model for Book Reviews (Intermediate)
Create a `BookReview` model with:
- `title: str` — the book title
- `rating: int` — use `Field()` to constrain between 1 and 5
- `genre: Literal[...]` — pick 4-5 genres (e.g., "Fiction", "Non-Fiction", "Sci-Fi", "Mystery", "Biography")
- `pros: list[str]` and `cons: list[str]` — what the reviewer liked/disliked

Then use `.parse()` to extract a structured review from a paragraph of text. Verify that the LLM returns the correct types and respects your constraints.

---

**What's Next?** In the next unit, we'll explore advanced agent frameworks like OpenAI's Agents SDK that handle these patterns for you automatically.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---